# SGDense-3DGS — segmentation-rendering performance branch

Tests the `perf/seg-render` branch:

1. **Build** the seg-aware rasterizer from this branch (with a guard against a stale install).
2. **Correctness** — the `dL_dalpha` seg-folding guard (`tests/test_seg_backward.py`).
3. **Perf** — a short SGDense run with the phase profiler, and the `--seg_render_interval K` lever.

See `docs/09-iteration-performance.md`.

> Runtime: **GPU**. Expects the Replica `office_0/Sequence_1` dataset (with
> `semantic_class/`) in the same Drive location as `Run_Experiments.ipynb`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Get the `perf/seg-render` branch

Parent repo **and** the seg-aware rasterizer submodule both need the branch. The
submodule is a separate repo, so we check it out explicitly and confirm the commit.

In [ ]:
import os

REPO_URL = "https://github.com/satyaba/gaussian-splatting.git"
BRANCH   = "perf/seg-render"
REPO_DIR = "/content/gaussian-splatting"
RAST_DIR = os.path.join(REPO_DIR, "submodules", "diff-gaussian-rasterization")

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --all -q
    !git -C {REPO_DIR} checkout -q {BRANCH}
    !git -C {REPO_DIR} pull --ff-only -q

os.chdir(REPO_DIR)

# Init submodules (rasterizer + simple-knn, incl. the rasterizer's nested glm).
!git submodule update --init --recursive submodules/diff-gaussian-rasterization submodules/simple-knn
# Make sure the rasterizer is on the perf branch tip (the recorded SHA should
# already be it, but this is the safety net).
!git -C {RAST_DIR} fetch -q origin {BRANCH}
!git -C {RAST_DIR} checkout -q origin/{BRANCH}
!git -C {RAST_DIR} submodule update --init --recursive

print("parent    :", end=" "); !git -C {REPO_DIR} log --oneline -1
print("rasterizer:", end=" "); !git -C {RAST_DIR} log --oneline -1
print("cwd:", os.getcwd())

## 3. GPU / CUDA check

In [ ]:
!nvidia-smi -L

import re, shutil, subprocess, warnings
import torch

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

def _nvcc_cuda():
    exe = shutil.which("nvcc")
    if exe is None:
        return None
    out = subprocess.run([exe, "--version"], capture_output=True, text=True).stdout
    m = re.search(r"release (\d+)\.(\d+)", out)
    return f"{m.group(1)}.{m.group(2)}" if m else None

torch_cuda, nvcc_cuda = torch.version.cuda, _nvcc_cuda()
print(f"  torch built for CUDA {torch_cuda} | nvcc {nvcc_cuda or 'NOT FOUND'}")
if nvcc_cuda is None:
    warnings.warn("nvcc not found - CUDA extensions cannot be built.")
elif torch_cuda and torch_cuda.split(".")[0] != nvcc_cuda.split(".")[0]:
    raise RuntimeError(f"CUDA major mismatch: torch {torch_cuda} vs nvcc {nvcc_cuda}.")

In [ ]:
!pip install -q plyfile opencv-python tqdm joblib lpips torchmetrics tensorboard

## 4. Configure paths

Same layout as `experiments/runs.py`: `<HOME_DIR>/Dataset/<dataset>` with
`images/`, `sparse/0/` and `semantic_class/`.

In [ ]:
import os
from pathlib import Path

HOME_DIR = "/content/drive/MyDrive/Magister/Thesis/Script"   # <-- adjust if different
DATASET_NAME = "office_0/Sequence_1"

DATASET_DIR = Path(HOME_DIR) / "Dataset" / DATASET_NAME
SEG_DIR = DATASET_DIR / "semantic_class"
OUT_ROOT = Path(HOME_DIR) / "output"

for p in (DATASET_DIR / "images", DATASET_DIR / "sparse" / "0", SEG_DIR):
    assert p.exists(), f"missing: {p}"
print("dataset:", DATASET_DIR)
print("seg    :", SEG_DIR, f"({len(list(SEG_DIR.glob('*.png')))} PNGs)")

## 5. Build the seg-aware rasterizer (required — CUDA changed)

**Uninstall first.** A stale install is the #1 cause of a `TypeError: unexpected
keyword argument 'do_seg'` inside `train.py`: `render()` imports
`GaussianRasterizationSettings` from the *installed* package, so the compiled
extension and its Python wrapper must be the ones built here. We then assert the
installed package actually exposes the new fields.

In [ ]:
# <cstdint> guard for newer CUDA/libstdc++ (idempotent; same patch the harness applies).
for f in ["submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h",
          "submodules/simple-knn/cuda_utils.h",
          "submodules/simple-knn/simple_knn.cu"]:
    if os.path.isfile(f):
        txt = open(f).read()
        if "include <cstdint>" not in txt:
            open(f, "w").write("#include <cstdint>\n" + txt)
            print("patched", f)

!pip uninstall -y -q diff-gaussian-rasterization
!pip install ./submodules/diff-gaussian-rasterization --no-build-isolation 2>&1 | tail -25
!pip install ./submodules/simple-knn --no-build-isolation 2>&1 | tail -3

In [ ]:
# Drop any already-imported rasterizer, then verify the INSTALLED build is ours.
import sys, importlib
for _m in [m for m in sys.modules if m.startswith("diff_gaussian_rasterization")]:
    del sys.modules[_m]
import diff_gaussian_rasterization as dgr
from pathlib import Path

print("installed from:", dgr.__file__)
sos = list(Path(dgr.__file__).parent.glob("_C*.so"))
assert sos, f"no compiled extension next to {dgr.__file__} - build failed"
print("extension:", sos[0].name)

fields = dgr.GaussianRasterizationSettings._fields
missing = [f for f in ("do_seg", "dL_dmean2D_photo") if f not in fields]
assert not missing, (
    f"STALE RASTERIZER INSTALL: missing settings fields {missing}. "
    f"Got: {fields}. Re-run cell 5 and read the build output above.")
print("OK: installed rasterizer exposes the perf/seg-render API")

## 6. Correctness sanity — seg still drives geometry

`tests/test_seg_backward.py` is the doc-01 §2 guard: a seg-only loss must produce
nonzero `xyz`/`opacity`/`scaling` gradients (via the shared `dL_dalpha`) and exactly
zero color gradients.

In [ ]:
!python -u tests/test_seg_backward.py

## 7. Phase profile — default (every iteration, `K=1`)

A short run with the profiler enabled, over a window just past the seg warmup
(`seg_warmup_iters=2000`), where the old second backward used to run. The helper
now captures the child output, so a failure prints `train.py`'s real error.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

ITERATIONS     = 2600
PROFILE_WINDOW = "2400:2599"

def run_sgdense(tag, extra=()):
    out_dir = OUT_ROOT / f"perf_{tag}"
    out_dir.mkdir(parents=True, exist_ok=True)
    profile_out = str(out_dir / "iter_profile.json")
    cmd = [sys.executable, "train.py",
           "-s", str(DATASET_DIR), "-m", str(out_dir),
           "--eval", "--disable_viewer", "--quiet",
           "--iterations", str(ITERATIONS),
           "--segmentation_path", str(SEG_DIR),
           "--num_segmentation_classes", "99",
           *extra]
    env = dict(os.environ,
               SGDENSE_PROFILE_ITERS=PROFILE_WINDOW,
               SGDENSE_PROFILE_OUT=profile_out)
    print(f"\n=== run {tag}: {' '.join(extra) or '(default)'} ===")
    print("$ " + " ".join(cmd))
    t0 = time.time()
    p = subprocess.run(cmd, cwd=REPO_DIR, env=env, capture_output=True, text=True)
    out = (p.stdout or "") + (p.stderr or "")
    if p.returncode != 0:
        print(f"\n!!! train.py exited {p.returncode}; last 80 output lines:\n")
        print("\n".join(out.splitlines()[-80:]))
        raise RuntimeError(f"train.py failed for {tag} (exit {p.returncode})")
    print("\n".join(out.splitlines()[-15:]))
    dt = time.time() - t0
    print(f"wall: {dt:.1f}s ({dt / ITERATIONS * 1000:.1f} ms/iter avg)")
    return profile_out, dt

def show_profile(path, title):
    g = json.load(open(path))["summary"]["groups"].get("all", {})
    print(f"\n{title}")
    print(f"  {'phase':<10} {'mean_ms':>9} {'p90_ms':>9}")
    for ph in ["render", "loss", "backward", "optimizer", "densify", "misc", "total"]:
        m = g.get(ph, {})
        if "mean_ms" in m:
            print(f"  {ph:<10} {m['mean_ms']:>9.2f} {m.get('p90_ms', float('nan')):>9.2f}")

json_k1, wall_k1 = run_sgdense("K1")
show_profile(json_k1, "K=1 (every iteration)")

## 8. Phase profile — seg-render interval `K=8`

Off-iterations skip the 32-channel seg compositing and the decoder, so they cost
≈ vanilla. **This changes training dynamics** (fewer decoder updates / seg→geometry
gradients) — treat it as an ablation, not a free win.

In [ ]:
json_k8, wall_k8 = run_sgdense("K8", extra=("--seg_render_interval", "8"))
show_profile(json_k8, "K=8 (seg every 8th iteration)")

print(f"\nWall-clock: K=1 {wall_k1:.1f}s  |  K=8 {wall_k8:.1f}s  "
      f"({wall_k1 / wall_k8:.2f}x)")

### Reading the numbers

- **`backward`** is now the single backward only (the old `autograd.grad` pass is gone).
- **`render`** drops, on average, by roughly `seg_extra * (K-1)/K` at `K=8`.
- **`optimizer` / `misc`** are unchanged.
- Sanity: PSNR shouldn't collapse at `K=1`; `K=8` is the ablation to watch.

## 9. (Optional) Full A/B against vanilla 3DGS

`experiments.profile` clones upstream 3DGS, rebuilds the stock rasterizer, runs both
methods, and prints the per-phase SGDense-vs-vanilla table. It **swaps the installed
rasterizer**, so re-run cell 5 afterward to switch back to the seg-aware one.

In [ ]:
# Longer, standard-config A/B. The 2600-iter run was CONFOUNDED: SGDense gates
# densification behind seg_warmup_iters=2000, so at 2600 it had ~6 densify events
# (11k Gaussians) vs vanilla's ~21 (203k). 8000 iters + the standard
# '--floater_action freeze' gives comparable populations; profile a post-warmup window.
!python -m experiments.profile --source "{DATASET_DIR}" --home-dir "{HOME_DIR}" \
    --iterations 8000 --profile-iters 7000:7199 --seg-num-classes 99 \
    --extra-sgdense "--floater_action freeze --densify_until_iter 25000"

### Compare Gaussian populations (must be similar for the phase table to mean anything)

In [ ]:
import json
HOME = HOME_DIR
for label, p in [("vanilla", f"{HOME}/output/profile_vanilla/iter_profile.json"),
                 ("sgdense", f"{HOME}/output/profile_sgdense/iter_profile.json")]:
    recs = json.load(open(p))["records"]
    ns = [r["meta"]["n_gaussians"] for r in recs.values()]
    print(f"{label:8s} n_gaussians  mean {sum(ns)/len(ns):,.0f}  min {min(ns):,}  max {max(ns):,}")